# Chapter 8 Simulations — Grid Search, Ground-Truth Evaluation & Judge Calibration

This notebook runs three evaluations against the three-agent processing pipeline defined in `my_agent/agent.py`:

1. **Grid Search** — three models × three temperatures × ten signal scenarios × five Monte Carlo simulations = 450 total runs. Ranked by F1 (judge-compliance metric). Writes the winning configuration to `best_config.json`.
2. **Ground-Truth Evaluation** — three expert-curated scenarios from `ground_truth.csv`. Pipeline output is compared against analyst-expected findings via Confidence Accuracy.
3. **Judge Calibration** — ten synthetic corroboration briefs with known-correct verdicts test whether the Processing Validation Judge correctly identifies corroboration rule violations, confidence inflation, and missed correlations.

**Why all three?** Grid search measures how well the Correlation Agent satisfies the judge. Ground-truth measures whether the brief matches what a human analyst would produce. Judge calibration measures whether the judge itself can be trusted. A lenient judge produces F1=100% on a brief with wrong confidence levels — only the ground-truth and calibration tests catch that.

## 1. Setup

Load environment variables and import the pipeline. `run_pipeline()` is imported directly from `my_agent/agent.py` — it constructs fresh agent instances per call and manages its own session state, so no manual session wiring is needed in the notebook.

In [6]:
import asyncio
import json
import os
import sys
import time
import statistics
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import HTML, display

from dotenv import load_dotenv

# Load API key — tries local .env first, then falls back to Colab Secrets.
load_dotenv()  # walks up the directory tree — finds root .env
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass  # Not in Colab or secret not configured

# Make my_agent importable from the chapter-08/ root.
sys.path.insert(0, str(Path(".").resolve()))

from my_agent.agent import run_pipeline, DEFAULT_MODEL, DEFAULT_TEMP, ProcessingVerdict
from my_agent.indicator_schemas import (
    SIGNAL_TYPES,
    SOURCE_CATEGORY_MAP,
    CONFIDENCE_LEVELS,
    CORROBORATION_RULES,
    TLP_LEVELS,
)

print(f"Default model  : {DEFAULT_MODEL} @ temp={DEFAULT_TEMP}")
print(f"Signal types   : {len(SIGNAL_TYPES)}")
print(f"Src categories : {len(SOURCE_CATEGORY_MAP)}")
print(f"Corr rules     : {len(CORROBORATION_RULES)}")

Default model  : gemini-2.5-flash @ temp=0.2
Signal types   : 10
Src categories : 24
Corr rules     : 5


## 2. Pipeline Helper

Wraps a single end-to-end run of `run_pipeline()` from `agent.py`. The agent constructs fresh instances per call — reusing instances across calls would raise "Agent already has a parent" from ADK's sub-agent registration.

The `run_scenario()` helper returns `(session_id, iterations, metrics)`. Metrics are computed by `compute_metrics()` defined in the next section.

## 3. Compute Metrics

Three judge-compliance metrics derived from the final iteration's structured verdict:

- **Precision** = `valid / (valid + unverified)` × 100 — fraction of confidence assignments that comply with corroboration rules
- **Coverage** = `valid / (valid + missing_critical)` × 100 — fraction of obvious cross-source correlations the agent actually made
- **F1** = harmonic mean of the two — convergence target is 100%

These are *judge-compliance* metrics. The ground-truth section provides the independent external reference.

In [7]:
def compute_metrics(iterations: list[dict]) -> dict:
    """Derive precision, coverage, F1, and iteration count from the final verdict."""
    final = iterations[-1]["verdict"]
    n_v = len(final.get("confirmed_valid", []))
    n_u = len(final.get("unverified", []))
    n_m = len(final.get("missing_critical", []))

    precision = (n_v / (n_v + n_u) * 100) if (n_v + n_u) else 100.0
    coverage  = (n_v / (n_v + n_m) * 100) if (n_v + n_m) else 100.0
    f1 = (2 * precision * coverage / (precision + coverage)
          if (precision + coverage) else 0.0)

    n_essential = sum(
        1 for s in final.get("confirmed_valid", [])
        if s.get("relevance") == "ESSENTIAL"
    )
    n_relevant = sum(
        1 for s in final.get("confirmed_valid", [])
        if s.get("relevance", "ESSENTIAL") in ("ESSENTIAL", "USEFUL")
    )
    relevance = (n_relevant / n_v * 100) if n_v else 100.0

    return {
        "precision":    round(precision, 1),
        "coverage":     round(coverage, 1),
        "f1":           round(f1, 1),
        "relevance":    round(relevance, 1),
        "iters":        len(iterations),
        "n_valid":      n_v,
        "n_unverified": n_u,
        "n_missing":    n_m,
        "n_essential":  n_essential,
        "passed":       final.get("verdict") == "PASS",
        "verdict":      final.get("verdict", "UNKNOWN"),
    }

In [8]:
async def run_scenario(
    collection_plan: str,
    threat_context: str = "",
    max_iterations: int = 3,
    model: str = DEFAULT_MODEL,
    temperature: float = DEFAULT_TEMP,
    verbose: bool = True,
) -> tuple[str, list[dict], dict]:
    """
    Run the pipeline on one scenario and return (session_id, iterations, metrics).

    Prints per-iteration output when verbose=True.
    """
    if verbose:
        print(f"Model: {model} @ temp={temperature}  max_iter={max_iterations}")
        print()

    session_id, iterations = await run_pipeline(
        collection_plan=collection_plan,
        max_iterations=max_iterations,
        threat_context=threat_context,
        processing_model=model,
        processing_temp=temperature,
    )

    if verbose:
        for it in iterations:
            n       = it["iteration"]
            verdict = it["verdict"]
            n_v     = len(verdict.get("confirmed_valid", []))
            n_u     = len(verdict.get("unverified", []))
            n_m     = len(verdict.get("missing_critical", []))
            label   = verdict.get("verdict", "UNKNOWN")
            print(f"{'─'*60}")
            print(f"Iteration {n} — {label}")
            print(f"  valid={n_v}  unverified={n_u}  missing={n_m}")
            print(f"  {verdict.get('summary', '')[:120]}")
            print()
            print("Corroboration Brief (first 1 500 chars):")
            print(it.get("corroboration_brief", "")[:1500])
            print()

        m = compute_metrics(iterations)
        print(f"{'='*60}")
        print(f"Final verdict : {m['verdict']} in {m['iters']} iteration(s)")
        print(f"Precision     : {m['precision']:.1f}%")
        print(f"Coverage      : {m['coverage']:.1f}%")
        print(f"F1            : {m['f1']:.1f}%")

    return session_id, iterations, compute_metrics(iterations)

## 4. Single-Scenario Demo

Runs the AiTM Session Hijacking — Full Chain scenario: the canonical threat for ApexCode's partner access environment. The collection plan below is the Stage 2 output the pipeline is designed to consume.

This demonstrates the full self-refining loop end-to-end: Correlation Agent drafts the brief, Processing Validation Judge emits a structured JSON verdict, Verification Agent applies corrections, and the loop exits on PASS via the Verification Agent’s escalation callback.

In [ ]:
# NOTE: REQUIRES API KEY - single pipeline run (up to ~9 model calls).
# The demo input is the canonical AiTM scenario exactly as stored in
# ground_truth.csv (scenario 1) - the same Stage 2 collection plan the
# ground-truth evaluation consumes.
import csv

with open("ground_truth.csv", newline="", encoding="utf-8") as _f:
    _demo = next(csv.DictReader(_f))

print(f"Scenario: {_demo['Scenario']}\n")
print(_demo["Collection_Plan"].strip()[:400] + "\n[... full plan passed to the pipeline ...]\n")

demo_session_id, demo_iterations, demo_metrics = await run_scenario(
    collection_plan=_demo["Collection_Plan"],
    threat_context=_demo["Threat_Context"],
    verbose=True,
)

## 5. Grid Search

The sweep covers three models × three temperatures × ten signal scenarios × five Monte Carlo simulations = 450 total runs, capped at 20 concurrent pipelines via `asyncio.Semaphore`. Each scenario exercises a distinct threat vector so the winning configuration generalises beyond the AiTM canonical case.

Ranked by average F1; ties broken by average iteration count (fewer iterations = faster convergence), then by average latency. The winning configuration is written to `best_config.json` and loaded by `agent.py` at import time for the `adk web` interface.

Where all configurations return F1 near 100%, the benchmark has hit a ceiling — the processing task is constrained enough by the corroboration rules that model capability is not the limiting factor. In that case cost and latency favour the cheapest passing configuration. Tune `CONCURRENCY_LIMIT` to approximately one-third of your per-minute API quota (each active pipeline slot generates roughly six sequential LLM calls).

In [ ]:
# ── Grid configuration ─────────────────────────────────────────────────────

MODELS = [
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-3.1-pro-preview",
]
TEMPERATURES     = [0.0, 0.5, 1.0]
N_SIMULATIONS    = 5
CONCURRENCY_LIMIT = 20
MAX_ITERATIONS   = 3
RESULTS_CSV      = "config_search_results.csv"
BEST_CONFIG_JSON = "my_agent/best_config.json"

# ── Ten signal scenarios ───────────────────────────────────────────────────
# Each provides a distinct collection plan so the grid tests generalisation
# across threat types, not just repeated AiTM runs.

SCENARIOS = [
    {
        "name": "AiTM Session Hijacking",
        "collection_plan": (
            "### 1. Internal\n"
            "- Okta System Log: user.session.start, user.authentication.auth_via_mfa\n"
            "- Microsoft Entra ID Sign-In Logs: interactiveUserSignIn\n"
            "- GitHub Audit Log: git.clone, personal_access_token.create\n"
            "- Proofpoint Email Security: click.permitted, impostor.detected\n\n"
            "### 2. External\n"
            "- HaveIBeenPwned API: partner domain breach monitoring\n"
            "- SpyCloud: session cookie stealer log monitoring\n"
            "- urlscan.io: phishing infrastructure detection\n\n"
            "### 3. TTPs\nAiTM via Evilginx2 (T1557). Priority Signal: session with no auth event."
        ),
        "threat_context": (
            "Partner accounts targeted via AiTM proxy phishing. "
            "Session cookies stolen; no new IdP event observed."
        ),
    },
    {
        "name": "Ransomware via Stolen VPN Credentials",
        "collection_plan": (
            "### 1. Internal\n"
            "- Palo Alto Networks Firewall (PAN-OS): THREAT events, lateral movement\n"
            "- CrowdStrike Falcon: DetectionSummaryEvent, ProcessRollup2\n"
            "- Microsoft Sentinel: correlated alerts across endpoints\n"
            "- Microsoft Entra ID Sign-In Logs: VPN authentication events\n\n"
            "### 2. External\n"
            "- SpyCloud: credential monitoring for VPN accounts\n"
            "- Abuse.ch ThreatFox: ransomware C2 IOC matching\n"
            "- AlienVault OTX (Open Threat Exchange): threat actor TTPs\n\n"
            "### 3. TTPs\nCredential theft + VPN access + ransomware deployment (T1486)."
        ),
        "threat_context": (
            "VPN credentials stolen via infostealer. "
            "Adversary uses legitimate VPN access to deploy ransomware."
        ),
    },
    {
        "name": "BEC with OAuth Consent Abuse",
        "collection_plan": (
            "### 1. Internal\n"
            "- Microsoft Entra ID Audit Logs: OAuth app consent grants\n"
            "- Microsoft 365 Unified Audit Log: mail access, delegation rules\n"
            "- Microsoft Entra ID Sign-In Logs: service principal sign-ins\n\n"
            "### 2. External\n"
            "- urlscan.io: phishing page hosting OAuth consent screens\n"
            "- VirusTotal: malicious OAuth app hash or domain\n\n"
            "### 3. TTPs\nOAuth consent phishing (T1528). Persistent mail access via delegated token."
        ),
        "threat_context": (
            "BEC via OAuth consent abuse. "
            "Adversary tricks user into granting mail.read permissions to rogue app."
        ),
    },
    {
        "name": "Insider Threat — Bulk Data Download",
        "collection_plan": (
            "### 1. Internal\n"
            "- Netskope CASB: bulk_download events, DLP policy triggers\n"
            "- GitHub Audit Log: git.clone, repo.access outside business hours\n"
            "- Microsoft 365 Unified Audit Log: SharePoint file bulk access\n"
            "- Okta System Log: session activity correlated with download events\n\n"
            "### 2. External\n"
            "- None applicable — insider threat is internal by definition.\n\n"
            "### 3. TTPs\nData staging and exfiltration (T1074, T1048). Departing employee pattern."
        ),
        "threat_context": (
            "Departing engineer with access to proprietary source code. "
            "Bulk clone and personal cloud upload pattern."
        ),
    },
    {
        "name": "API Key Exposure on Public GitHub",
        "collection_plan": (
            "### 1. Internal\n"
            "- GitHub Audit Log: public repo creation, secret scanning alerts\n"
            "- AWS CloudTrail: AssumeRole, GetCallerIdentity from unknown IP\n"
            "- CrowdStrike Falcon: process events on CI/CD runner hosts\n\n"
            "### 2. External\n"
            "- GitHub Public Events API: secret scanning webhook alerts\n"
            "- GreyNoise: IP reputation for API caller\n"
            "- HaveIBeenPwned API: developer email in breach data\n\n"
            "### 3. TTPs\nCredential exposure via public repo (T1552.001). Cloud account takeover risk."
        ),
        "threat_context": (
            "AWS API key committed to public GitHub repo. "
            "Automated scanners harvest keys within minutes of exposure."
        ),
    },
    {
        "name": "Supply Chain Compromise via Malicious Package",
        "collection_plan": (
            "### 1. Internal\n"
            "- CrowdStrike Falcon: suspicious process spawned from npm/pip install\n"
            "- Palo Alto Networks Firewall (PAN-OS): outbound C2 from build servers\n"
            "- GitHub Audit Log: dependency file changes in critical repos\n\n"
            "### 2. External\n"
            "- Abuse.ch ThreatFox: C2 IP matching known supply chain actor infrastructure\n"
            "- VirusTotal: malicious package hash\n"
            "- AlienVault OTX (Open Threat Exchange): supply chain campaign IOCs\n\n"
            "### 3. TTPs\nDependency confusion or typosquatting (T1195.001)."
        ),
        "threat_context": (
            "Malicious npm package with near-identical name to internal package. "
            "Installs backdoor on developer workstations."
        ),
    },
    {
        "name": "Credential Stuffing Against Okta",
        "collection_plan": (
            "### 1. Internal\n"
            "- Okta System Log: user.authentication.auth_via_mfa failures, rate-limit events\n"
            "- Microsoft Entra ID Sign-In Logs: failed sign-ins from distributed IPs\n"
            "- Palo Alto Networks Firewall (PAN-OS): high-volume auth traffic from ASNs\n\n"
            "### 2. External\n"
            "- GreyNoise: IP tagging for credential stuffing bots\n"
            "- SpyCloud: credential lists matching Okta usernames\n"
            "- HaveIBeenPwned API: breach hits for the targeted user population\n\n"
            "### 3. TTPs\nCredential stuffing (T1110.004). Distributed low-and-slow pattern."
        ),
        "threat_context": (
            "Automated credential stuffing campaign against Okta tenant. "
            "Distributed across residential proxies to evade rate limiting."
        ),
    },
    {
        "name": "Phishing Kit Targeting Partner SSO",
        "collection_plan": (
            "### 1. Internal\n"
            "- Proofpoint Email Security: click.permitted on suspicious URLs\n"
            "- Cisco Umbrella: DNS requests to newly registered domains\n"
            "- Okta System Log: authentication events from unexpected geos\n\n"
            "### 2. External\n"
            "- urlscan.io: phishing page mimicking partner SSO portal\n"
            "- dnstwist: typosquat domains targeting partner company name\n"
            "- PhishTank / CheckPhish: phishing kit classification\n\n"
            "### 3. TTPs\nSpearphishing link (T1566.002). SSO credential harvesting."
        ),
        "threat_context": (
            "Phishing campaign targeting partner employees with SSO lookalike pages. "
            "Harvested credentials used to access ApexCode shared environments."
        ),
    },
    {
        "name": "Privileged Access Escalation via PAM Bypass",
        "collection_plan": (
            "### 1. Internal\n"
            "- CyberArk Privileged Access Manager: PSM.Session.Start anomalies\n"
            "- Okta System Log: group membership changes outside change window\n"
            "- AWS CloudTrail: AssumeRole to admin role from non-corporate IP\n"
            "- Microsoft Sentinel: correlated privilege escalation alerts\n\n"
            "### 2. External\n"
            "- SpyCloud: credentials matching privileged account usernames\n\n"
            "### 3. TTPs\nPrivilege escalation via PAM session (T1078.004)."
        ),
        "threat_context": (
            "Compromised service account used to check out privileged credentials "
            "from PAM vault outside approved workflow."
        ),
    },
    {
        "name": "Dark Web Access Broker Listing",
        "collection_plan": (
            "### 1. Internal\n"
            "- Okta System Log: authentication from IPs matching dark web exit nodes\n"
            "- Microsoft Entra ID Sign-In Logs: impossible travel events\n"
            "- Palo Alto Networks Firewall (PAN-OS): outbound traffic to Tor exit nodes\n\n"
            "### 2. External\n"
            "- IntelligenceX: dark web forum monitoring for ApexCode mentions\n"
            "- SpyCloud: access broker credential listings\n"
            "- GreyNoise: Tor exit node IP tagging\n\n"
            "### 3. TTPs\nInitial access broker sale of network access (T1650)."
        ),
        "threat_context": (
            "Access broker listing for 'SaaS dev company' matching ApexCode profile. "
            "Listing appeared 48 hours before anomalous auth events."
        ),
    },
]

print(f"Grid: {len(MODELS)} models × {len(TEMPERATURES)} temps "
      f"× {len(SCENARIOS)} scenarios × {N_SIMULATIONS} sims "
      f"= {len(MODELS) * len(TEMPERATURES) * len(SCENARIOS) * N_SIMULATIONS} total runs")

In [ ]:
# NOTE: REQUIRES API KEY only if you execute run_grid() - 450 runs x ~6 calls
# = ~2,700 model calls. As shipped, this cell only loads config_search_results.csv.

import math

# ── Helpers ────────────────────────────────────────────────────────────────

# Display-name -> shipped CSV schema (inverse of the load path's _col_map).
_DISPLAY_TO_CSV = {
    "Model":             "model",
    "Temperature":       "temperature",
    "avg F1 (%)":        "avg_f1",
    "F1 Std Dev":        "avg_f1_std",
    "avg Precision (%)": "avg_precision",
    "avg Coverage (%)":  "avg_coverage",
    "avg Iters":         "avg_iters",
    "Pass Rate (%)":     "pass_rate",
    "avg Latency (s)":   "avg_latency_s",
}

def _agg(sim_results: list[dict]) -> dict:
    """Aggregate N simulation results into mean/std metrics for one config × scenario."""
    def _mean(key):
        vals = [r[key] for r in sim_results if r.get(key) is not None]
        return statistics.mean(vals) if vals else 0.0

    def _std(key):
        vals = [r[key] for r in sim_results if r.get(key) is not None]
        return statistics.stdev(vals) if len(vals) > 1 else 0.0

    return {
        "avg_f1":        _mean("f1"),
        "std_f1":        _std("f1"),
        "avg_precision": _mean("precision"),
        "avg_coverage":  _mean("coverage"),
        "avg_iters":     _mean("iters"),
        "avg_latency":   _mean("latency_s"),
        "pass_rate":     sum(1 for r in sim_results if r.get("passed")) / len(sim_results) * 100,
        "n_sims":        len(sim_results),
    }


async def _run_single_sim(
    sem: asyncio.Semaphore,
    scenario: dict,
    model: str,
    temperature: float,
    retry_limit: int = 3,
) -> dict:
    """Run one simulation with concurrency control and exponential backoff retry."""
    async with sem:
        for attempt in range(retry_limit):
            try:
                t0 = time.monotonic()
                _, iterations = await run_pipeline(
                    collection_plan=scenario["collection_plan"],
                    max_iterations=MAX_ITERATIONS,
                    threat_context=scenario.get("threat_context", ""),
                    processing_model=model,
                    processing_temp=temperature,
                )
                m = compute_metrics(iterations)
                m["latency_s"] = round(time.monotonic() - t0, 1)
                return m
            except Exception as exc:
                if attempt < retry_limit - 1:
                    wait = (2 ** attempt) * 5
                    print(f"    Retry {attempt + 1}/{retry_limit - 1} for "
                          f"{model} T={temperature} '{scenario['name']}': {exc} — waiting {wait}s")
                    await asyncio.sleep(wait)
                else:
                    print(f"    FAILED after {retry_limit} attempts: {exc}")
                    return {"f1": 0.0, "precision": 0.0, "coverage": 0.0,
                            "iters": MAX_ITERATIONS, "latency_s": 0.0, "passed": False,
                            "verdict": "FAIL"}


async def run_grid() -> pd.DataFrame:
    """
    Execute the full 3 × 3 × 10 × 5 grid and return a ranked summary DataFrame.
    Results are checkpointed to RESULTS_CSV after each (model, temperature) pair;
    on completion the ranked grid is re-written and the winning configuration
    is saved to BEST_CONFIG_JSON, which agent.py loads at import time.
    """
    sem = asyncio.Semaphore(CONCURRENCY_LIMIT)
    all_rows = []

    total_configs = len(MODELS) * len(TEMPERATURES)
    config_idx = 0

    for model in MODELS:
        for temp in TEMPERATURES:
            config_idx += 1
            print(f"\n[{config_idx}/{total_configs}] {model} @ temp={temp}")

            # Build all simulation tasks for this config across all scenarios.
            tasks = [
                _run_single_sim(sem, sc, model, temp)
                for sc in SCENARIOS
                for _ in range(N_SIMULATIONS)
            ]
            results = await asyncio.gather(*tasks)

            # Group by scenario.
            idx = 0
            sc_aggs = []
            for sc in SCENARIOS:
                sc_results = list(results[idx : idx + N_SIMULATIONS])
                idx += N_SIMULATIONS
                agg = _agg(sc_results)
                sc_aggs.append(agg)
                print(f"  {sc['name'][:40]:40s}  "
                      f"F1={agg['avg_f1']:.1f}%  "
                      f"pass={agg['pass_rate']:.0f}%  "
                      f"iters={agg['avg_iters']:.2f}")

            # Aggregate across scenarios to get config-level metrics.
            def _cfg_mean(key):
                return statistics.mean(a[key] for a in sc_aggs)

            row = {
                "Model":             model,
                "Temperature":       temp,
                "avg F1 (%)":        round(_cfg_mean("avg_f1"), 1),
                "F1 Std Dev":        round(
                    math.sqrt(sum(a["std_f1"] ** 2 for a in sc_aggs) / len(sc_aggs)), 1
                ),
                "avg Precision (%)": round(_cfg_mean("avg_precision"), 1),
                "avg Coverage (%)":  round(_cfg_mean("avg_coverage"), 1),
                "avg Iters":         round(_cfg_mean("avg_iters"), 2),
                "Pass Rate (%)":     round(_cfg_mean("pass_rate"), 1),
                "avg Latency (s)":   round(_cfg_mean("avg_latency"), 1),
            }
            all_rows.append(row)

            # Checkpoint partial results after each (model, temperature) pair.
            pd.DataFrame(all_rows).rename(columns=_DISPLAY_TO_CSV).to_csv(
                RESULTS_CSV, index=False
            )

    df = pd.DataFrame(all_rows)
    df = df.sort_values(
        ["avg F1 (%)", "avg Iters", "avg Latency (s)"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    df.insert(0, "Rank", range(1, len(df) + 1))

    # Persist the ranked grid (shipped CSV schema) and the winning config,
    # which agent.py loads at import time.
    df.drop(columns=["Rank"]).rename(columns=_DISPLAY_TO_CSV).to_csv(
        RESULTS_CSV, index=False
    )
    best = df.iloc[0]
    Path(BEST_CONFIG_JSON).write_text(
        json.dumps({"model": best["Model"], "temperature": float(best["Temperature"])})
    )
    print(f"\nWrote {RESULTS_CSV} and {BEST_CONFIG_JSON} "
          f"(winner: {best['Model']} @ temp={best['Temperature']})")
    return df


# ── Load pre-computed results from CSV ───────────────────────────────────
# Run run_grid() above to re-run the full grid search.
df_grid = pd.read_csv(RESULTS_CSV)
# Rename snake_case CSV columns to the display names expected by style functions
_col_map = {
    "model":         "Model",
    "temperature":   "Temperature",
    "avg_f1":        "avg F1 (%)",
    "avg_f1_std":    "F1 Std Dev",
    "avg_precision": "avg Precision (%)",
    "avg_coverage":  "avg Coverage (%)",
    "avg_iters":     "avg Iters",
    "pass_rate":     "Pass Rate (%)",
    "avg_latency_s": "avg Latency (s)",
}
df_grid = df_grid.rename(columns={k: v for k, v in _col_map.items() if k in df_grid.columns})
df_grid = df_grid.sort_values(
    ["avg F1 (%)", "avg Iters", "avg Latency (s)"], ascending=[False, True, True]
).reset_index(drop=True)
if "Rank" not in df_grid.columns:
    df_grid.insert(0, "Rank", range(1, len(df_grid) + 1))
print(f"Loaded {len(df_grid)} configurations from {RESULTS_CSV}")
df_grid


In [ ]:
### Grid search results — styled ranked table

def _style_grid(df: pd.DataFrame):
    def highlight_best(row):
        return ["background-color: #dcfce7; font-weight: bold" if row["★"] == "★"
                else "" for _ in row]

    def color_f1(val):
        if val >= 100.0: return "color: #15803d; font-weight: 600"
        if val >= 99.5:  return "color: #ca8a04"
        return "color: #dc2626"

    def color_pass(val):
        if val >= 100: return "color: #15803d; font-weight: 600"
        if val >= 96:  return "color: #ca8a04"
        return "color: #dc2626"

    display_df = df.copy()
    display_df.insert(0, "★", display_df["Rank"].apply(lambda r: "★" if r == 1 else ""))
    display_df = display_df.drop(columns=["Rank"])
    display_df = display_df.rename(columns={
        "avg F1 (%)":        "avg F1 %",
        "F1 Std Dev":        "± std",
        "avg Precision (%)": "Precision %",
        "avg Coverage (%)":  "Coverage %",
        "avg Iters":         "Iters",
        "Pass Rate (%)":     "Pass %",
        "avg Latency (s)":   "Latency (s)",
    })

    return (
        display_df.style
        .apply(highlight_best, axis=1)
        .map(color_f1,   subset=["avg F1 %"])
        .map(color_pass, subset=["Pass %"])
        .format({
            "avg F1 %":    "{:.1f}",
            "± std":       "{:.1f}",
            "Precision %": "{:.1f}",
            "Coverage %":  "{:.1f}",
            "Iters":       "{:.2f}",
            "Pass %":      "{:.0f}",
            "Latency (s)": "{:.1f}",
        })
        .set_caption(
            f"Config search — {len(MODELS)} models × {len(TEMPERATURES)} temps "
            f"× {len(SCENARIOS)} scenarios × {N_SIMULATIONS} sims = "
            f"{len(MODELS)*len(TEMPERATURES)*len(SCENARIOS)*N_SIMULATIONS} total runs. "
            "★ = selected configuration written to best_config.json."
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.85rem"), ("color", "#64748b"),
                       ("padding-bottom", "8px"), ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.85rem"), ("padding", "8px 12px"),
                       ("border-bottom", "1px solid #f1f5f9")]},
        ])
        .hide(axis="index")
    )

_style_grid(df_grid)

### Grid search results — aggregated by temperature

Average F1, precision, coverage, and pass rate grouped by temperature across all models and scenarios. Identifies whether temperature is the dominant factor for this pipeline.

In [ ]:
# ── Aggregated by temperature ──────────────────────────────────────────────
df_by_temp = (
    df_grid
    .groupby("Temperature", as_index=False)
    .agg(
        avg_f1        =("avg F1 (%)",        "mean"),
        avg_precision =("avg Precision (%)", "mean"),
        avg_coverage  =("avg Coverage (%)",  "mean"),
        avg_iters     =("avg Iters",         "mean"),
        pass_rate     =("Pass Rate (%)",     "mean"),
        avg_latency   =("avg Latency (s)",   "mean"),
    )
    .sort_values("avg_f1", ascending=False)
    .reset_index(drop=True)
)

df_by_temp.columns = [
    "Temperature", "avg F1 %", "Precision %", "Coverage %",
    "Iters", "Pass %", "Latency (s)",
]


def _style_temp_agg(df: pd.DataFrame):
    def color_f1(val):
        if val >= 100.0: return "color: #15803d; font-weight: 600"
        if val >= 99.5:  return "color: #ca8a04"
        return "color: #dc2626"

    return (
        df.style
        .map(color_f1, subset=["avg F1 %"])
        .format({
            "avg F1 %":    "{:.1f}",
            "Precision %": "{:.1f}",
            "Coverage %":  "{:.1f}",
            "Iters":       "{:.2f}",
            "Pass %":      "{:.0f}",
            "Latency (s)": "{:.1f}",
        })
        .set_caption("Grid search — aggregated by temperature (mean across all models and scenarios)")
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.85rem"), ("color", "#64748b"),
                       ("padding-bottom", "8px"), ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.85rem"), ("padding", "8px 12px"),
                       ("border-bottom", "1px solid #f1f5f9")]},
        ])
        .hide(axis="index")
    )


_style_temp_agg(df_by_temp)

### F1 distribution by model and temperature

Average F1 score for each model × temperature combination. Where all cells show 100%, the benchmark has hit a ceiling — the corroboration rules fully constrain the output space and model capability is not the limiting factor. In that case, cost and latency favour the cheapest passing configuration.

In [ ]:
# ── F1 pivot: Model × Temperature ─────────────────────────────────────────
df_pivot = df_grid.pivot_table(
    index="Model",
    columns="Temperature",
    values="avg F1 (%)",
    aggfunc="mean",
).round(1)

df_pivot.columns = [f"T={c}" for c in df_pivot.columns]
df_pivot.index.name = "Model"


def _style_f1_pivot(df: pd.DataFrame):
    def color_cell(val):
        if pd.isna(val):    return ""
        if val >= 100.0:    return "color: #15803d; font-weight: 600"
        if val >= 99.5:     return "color: #ca8a04"
        return "color: #dc2626"

    temp_cols = df.columns.tolist()
    return (
        df.style
        .map(color_cell, subset=temp_cols)
        .format("{:.1f}", subset=temp_cols, na_rep="–")
        .set_caption(
            "avg F1 % per Model × Temperature — green ≥ 100%, amber ≥ 99.5%, red < 99.5%. "
            "All-green grid = ceiling effect; select on cost/latency."
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.85rem"), ("color", "#64748b"),
                       ("padding-bottom", "8px"), ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.85rem"), ("padding", "8px 12px"),
                       ("border-bottom", "1px solid #f1f5f9"), ("text-align", "center")]},
        ])
    )


_style_f1_pivot(df_pivot)

## 6. Ground-Truth Evaluation

Loads three expert-curated scenarios from `ground_truth.csv` and runs the full pipeline against each. Output is compared to analyst-expected findings using **Confidence Accuracy** — the fraction of overlapping findings where the pipeline’s relevance rating is consistent with the expected confidence level (HIGH → ESSENTIAL, MEDIUM → ESSENTIAL/USEFUL, LOW → USEFUL/TANGENTIAL).

Threshold: Confidence Accuracy ≥ 70%.

Matching is decoupled from the judge verdict — signal type IDs are extracted from confirmed finding text and matched against ground truth keys. A well-calibrated judge can still fail the ground-truth check if the Correlation Agent misses key correlations. That is the failure mode this section is designed to surface.

In [9]:
import statistics

# ── Ground truth — load from CSV ──────────────────────────────────────────────
import pandas as pd

gt_df = pd.read_csv("ground_truth.csv")

GROUND_TRUTH = []
for _, row in gt_df.iterrows():
    GROUND_TRUTH.append({
        "name":                 row["Scenario"],
        "collection_plan":      row["Collection_Plan"],
        "threat_context":       row["Threat_Context"],
        "expected_findings":    json.loads(row["Expected_Findings"]),
        "expected_key_finding": row["Expected_Key_Finding"],
        "expected_attribution": row["Expected_Attribution"],
    })

print(f"Loaded {len(GROUND_TRUTH)} ground truth scenarios from ground_truth.csv")
for gt in GROUND_TRUTH:
    print(f"  {gt['name']} — {len(gt['expected_findings'])} expected findings")

# ── Matching and comparison functions ──────────────────────────────────────────

_SIGNAL_TYPE_IDS = set(SIGNAL_TYPES.keys())

_CONFIDENCE_TO_EXPECTED_RELEVANCE = {
    "HIGH":   {"ESSENTIAL"},
    "MEDIUM": {"ESSENTIAL", "USEFUL"},
    "LOW":    {"USEFUL", "TANGENTIAL"},
}


def _extract_signal_types(text: str) -> set[str]:
    """Extract known signal type IDs from text via ID and space-separated match."""
    text_lower = text.lower()
    found = set()
    for sig_id in _SIGNAL_TYPE_IDS:
        if sig_id in text_lower or sig_id.replace("_", " ") in text_lower:
            found.add(sig_id)
    return found


def extract_findings_from_verdict(verdict: dict) -> list[dict]:
    """Extract confirmed findings with signal types extracted from source text."""
    findings = []
    for item in verdict.get("confirmed_valid", []):
        source_text = item.get("source", "")
        findings.append({
            "source":       source_text,
            "relevance":    item.get("relevance", "UNKNOWN"),
            "signal_types": _extract_signal_types(source_text),
        })
    return findings


def _match_gt_key(gt_key: str, pipeline_findings: list[dict]) -> dict | None:
    """Match a ground truth key to the best-overlapping pipeline finding by signal type."""
    gt_types = _extract_signal_types(gt_key)
    if not gt_types:
        return None
    best, best_n = None, 0
    for pf in pipeline_findings:
        n = len(gt_types & pf.get("signal_types", set()))
        if n > best_n:
            best_n, best = n, pf
    return best if best_n > 0 else None


def compare_to_ground_truth(pipeline_findings: list[dict], gt_findings: dict) -> dict:
    """Compare pipeline findings to ground truth. Returns overlap, missed, extra, and confidence_accuracy."""
    gt_keys = list(gt_findings.keys())
    overlap, missed, conf_details = [], [], []
    conf_correct = conf_total = 0

    for gt_key in gt_keys:
        match = _match_gt_key(gt_key, pipeline_findings)
        if match:
            overlap.append(gt_key)
            exp_conf   = gt_findings[gt_key].get("confidence", "UNKNOWN")
            actual_rel = match.get("relevance", "UNKNOWN")
            correct    = actual_rel in _CONFIDENCE_TO_EXPECTED_RELEVANCE.get(exp_conf, set())
            conf_total += 1
            conf_correct += int(correct)
            conf_details.append({
                "finding":             gt_key,
                "expected_confidence": exp_conf,
                "pipeline_relevance":  actual_rel,
                "correct":             correct,
            })
        else:
            missed.append(gt_key)

    all_gt_types = set()
    for gk in gt_keys:
        all_gt_types |= _extract_signal_types(gk)
    extra = [
        pf["source"] for pf in pipeline_findings
        if not (pf.get("signal_types", set()) & all_gt_types)
    ]

    conf_acc = conf_correct / conf_total * 100 if conf_total > 0 else 0.0

    return {
        "overlap":             overlap,
        "missed":              missed,
        "extra":               extra,
        "confidence_accuracy": round(conf_acc, 1),
        "confidence_details":  conf_details,
        "gt_count":            len(gt_keys),
        "pipeline_count":      len(pipeline_findings),
    }


Loaded 3 ground truth scenarios from ground_truth.csv
  AiTM Session Hijacking — Full Chain — 3 expected findings
  Ransomware via Stolen VPN Credentials — 2 expected findings
  BEC with OAuth Consent Abuse — 2 expected findings


In [ ]:
# NOTE: REQUIRES API KEY for a fresh run - 3 scenarios x up to 3 iterations. With
# the shipped gt_checkpoint.json complete, this cell replays the recorded run and
# makes no API calls.

import asyncio
import json
from pathlib import Path
from tqdm.notebook import tqdm

import pandas as pd
import statistics
from IPython.display import display

GT_CHECKPOINT   = "gt_checkpoint.json"
GT_RESULTS_JSON = "ground_truth_results.json"
CONF_THRESHOLD  = 70.0


def _load_checkpoint() -> dict:
    if Path(GT_CHECKPOINT).exists():
        data = json.loads(Path(GT_CHECKPOINT).read_text())
        print(f"Checkpoint: {len(data)} scenario(s) already done: {list(data.keys())}")
        return data
    return {}


def _save_checkpoint(completed: dict) -> None:
    Path(GT_CHECKPOINT).write_text(json.dumps(completed, indent=2))


async def evaluate_ground_truth() -> list[dict]:
    completed = _load_checkpoint()
    results   = list(completed.values())
    pending   = [gt for gt in GROUND_TRUTH if gt["name"] not in completed]

    if not pending:
        print("All scenarios completed — loaded from checkpoint.")
        return results

    print(f"{len(pending)} scenario(s) remaining.\n")

    for gt in tqdm(pending, desc="Scenarios"):
        print(f"  Running: {gt['name']} ...")
        _, iterations = await run_pipeline(
            collection_plan=gt["collection_plan"],
            max_iterations=3,
            threat_context=gt.get("threat_context", ""),
        )

        final_verdict = iterations[-1]["verdict"]
        findings      = extract_findings_from_verdict(final_verdict)
        comparison    = compare_to_ground_truth(findings, gt["expected_findings"])
        n_iters       = len(iterations)
        verdict_label = final_verdict.get("verdict", "UNKNOWN")

        print(f"  Verdict  : {verdict_label} in {n_iters} iteration(s)")
        print(f"  Findings : pipeline={comparison['pipeline_count']}  gt={comparison['gt_count']}")
        print(f"  Overlap  : {len(comparison['overlap'])}  "
              f"Missed: {len(comparison['missed'])}  "
              f"Extra: {len(comparison['extra'])}")
        print(f"  Confidence Accuracy: {comparison['confidence_accuracy']:.0f}%")

        if comparison["missed"]:
            print("  Missed findings:")
            for key in comparison["missed"]:
                desc = gt["expected_findings"][key]["description"]
                print(f"    - {key}: {desc}")

        for cd in comparison["confidence_details"]:
            mark = "+" if cd["correct"] else "-"
            print(f"    {mark} {cd['finding'][:60]}: "
                  f"expected={cd['expected_confidence']} "
                  f"got={cd['pipeline_relevance']}")
        print()

        row = {
            "name":                gt["name"],
            "pipeline_count":      comparison["pipeline_count"],
            "gt_count":            comparison["gt_count"],
            "overlap":             len(comparison["overlap"]),
            "missed":              len(comparison["missed"]),
            "extra":               len(comparison["extra"]),
            "confidence_accuracy": comparison["confidence_accuracy"],
            "verdict":             verdict_label,
            "n_iters":             n_iters,
            "confidence_details":  comparison["confidence_details"],
        }
        results.append(row)
        completed[gt["name"]] = row
        _save_checkpoint(completed)
        if gt is not pending[-1]:
            print("  Cooling down 90s before next scenario (quota window reset)...")
            await asyncio.sleep(90)

    Path(GT_RESULTS_JSON).write_text(
        json.dumps(results, indent=2, default=list), encoding="utf-8"
    )
    print(f"Results saved → {GT_RESULTS_JSON}")
    return results


gt_results = await evaluate_ground_truth()

# ── Summary table ──────────────────────────────────────────────────────────

def _style_gt(rows: list[dict]):
    def row_color(row):
        color = "#dcfce7" if row["Conf Acc %"] >= CONF_THRESHOLD else "#fee2e2"
        return [f"background-color: {color}"] * len(row)

    def verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
        }.get(val, "")

    df = pd.DataFrame([{
        "Scenario":   r["name"],
        "GT":         r["gt_count"],
        "Pipeline":   r["pipeline_count"],
        "Overlap":    r["overlap"],
        "Missed":     r["missed"],
        "Extra":      r["extra"],
        "Conf Acc %": r["confidence_accuracy"],
        "Verdict":    r["verdict"],
        "Iters":      r["n_iters"],
    } for r in rows])

    avg_c  = df["Conf Acc %"].mean()
    status = "PASS ✓" if avg_c >= CONF_THRESHOLD else "FAIL ✗"

    _styler = df.style.apply(row_color, axis=1)
    _map = _styler.map
    return (
        _map(verdict_color, subset=["Verdict"])
        .format({"Conf Acc %": "{:.0f}%"})
        .set_caption(
            f"Ground-Truth Evaluation — avg Confidence Accuracy={avg_c:.0f}% "
            f"(threshold {CONF_THRESHOLD:.0f}%) — {status}"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.85rem"), ("padding", "8px 12px"),
                       ("border-bottom", "1px solid #f1f5f9")]},
        ])
        .hide(axis="index")
    )

display(_style_gt(gt_results))

# ── Pass/fail summary ───────────────────────────────────────────────────
avg_c = statistics.mean(r["confidence_accuracy"] for r in gt_results)
print()
print(f"avg Confidence Accuracy : {avg_c:.0f}%  (threshold {CONF_THRESHOLD:.0f}%)"
      f"  {'PASS' if avg_c >= CONF_THRESHOLD else 'FAIL'}")


## 7. Judge Calibration

Ten synthetic corroboration briefs with known-correct verdicts test whether the Processing Validation Judge correctly identifies rule violations. The cases cover five failure modes:

- **Valid brief** — all rules satisfied; judge should PASS
- **Confidence inflation** — HIGH assigned from only 2 signals (requires 3+)
- **CR-001** — same source category (edr_xdr + edr_xdr) claimed as corroboration
- **CR-002** — signals 30 days apart claimed as full corroboration (exceeds temporal proximity tiers)
- **CR-003** — conflicting geographic indicators treated as corroboration rather than flagged
- **CR-004** — signal pair not in each other's `corroborates_with` lists
- **CR-005** — attribution to named actor without an `infrastructure_match`
- **Missed correlation** — two corroboratable signals listed in isolation (PARTIAL expected)
- **Comprehensive brief** — 5 signals, all rules satisfied, multi-finding PASS

All 10 cases are scorable — unlike Chapter 7's judge_eval where 2 cases had no expected verdict. Accuracy threshold: 80% (8/10 must match on both verdict and violation count).

In [ ]:
# NOTE: REQUIRES API KEY - judge calibration: 10 model calls.

import pandas as pd
from IPython.display import display

from my_agent.judge_eval import TEST_CASES, run_judge_on_brief, _score_case

JUDGE_ACCURACY_THRESHOLD = 80.0

print("=" * 70)
print("JUDGE CALIBRATION — Processing Stage")
print(f"Test cases : {len(TEST_CASES)}  (all scorable)")
print(f"Threshold  : {JUDGE_ACCURACY_THRESHOLD:.0f}%")
print("=" * 70)

eval_rows   = []
total_pass  = total_fail = total_checks = schema_failures = 0

for i, case in enumerate(TEST_CASES, 1):
    print(f"\n{'─'*60}")
    print(f"Case {i}/{len(TEST_CASES)}: {case['name']}")
    print(f"  {case['description']}")

    result = await run_judge_on_brief(case["corroboration_brief"])

    if not result["valid"]:
        print(f"  SCHEMA FAILURE: {result['error'][:200]}")
        schema_failures += 1
        total_fail      += 1
        total_checks    += 1
        eval_rows.append({
            "name":                case["name"],
            "expected_verdict":    case.get("expected_verdict", "–"),
            "actual_verdict":      "ERROR",
            "expected_violations": case.get("expected_violations", "–"),
            "actual_violations":   "–",
            "verdict_correct":     False,
            "violations_correct":  False,
            "passed":              False,
            "summary":             result.get("error", "")[:80],
        })
        continue

    verdict = result["verdict"]
    scores  = _score_case(case, result)

    n_p = len(scores["passed_checks"])
    n_f = len(scores["failed_checks"])
    total_pass   += n_p
    total_fail   += n_f
    total_checks += n_p + n_f

    actual_verdict     = verdict.get("verdict", "UNKNOWN")
    actual_violations  = len(verdict.get("unverified", []))
    verdict_correct    = actual_verdict == case.get("expected_verdict")
    violations_correct = actual_violations == case.get("expected_violations")

    print(f"  Verdict  : {actual_verdict}  "
          f"(expected={case.get('expected_verdict', '–')})"
          f"  {'✓' if verdict_correct else '✗'}")
    print(f"  Unverified: {actual_violations}  "
          f"(expected={case.get('expected_violations', '–')})"
          f"  {'✓' if violations_correct else '✗'}")
    for check in scores["passed_checks"]:
        print(f"  + {check}")
    for check in scores["failed_checks"]:
        print(f"  - {check}")

    eval_rows.append({
        "name":                case["name"],
        "expected_verdict":    case.get("expected_verdict", "–"),
        "actual_verdict":      actual_verdict,
        "expected_violations": case.get("expected_violations", "–"),
        "actual_violations":   actual_violations,
        "verdict_correct":     verdict_correct,
        "violations_correct":  violations_correct,
        "passed":              n_f == 0,
        "summary":             verdict.get("summary", "")[:100],
    })

# ── Results summary ────────────────────────────────────────────────────────
accuracy = (total_pass / total_checks * 100) if total_checks > 0 else 0
status   = "PASS ✓" if accuracy >= JUDGE_ACCURACY_THRESHOLD else "FAIL ✗"
print(f"\n{'='*70}")
print(f"Checks passed : {total_pass}/{total_checks}  ({total_fail} failed, "
      f"{schema_failures} schema failure(s))")
print(f"Accuracy      : {accuracy:.0f}%  (threshold {JUDGE_ACCURACY_THRESHOLD:.0f}%) — {status}")
print("=" * 70)

# ── Styled table ───────────────────────────────────────────────────────────

def _style_judge_eval(rows: list[dict]) -> "pd.io.formats.style.Styler":
    def row_color(row):
        return (["background-color: #dcfce7"] * len(row) if row["Result"] == "✓"
                else ["background-color: #fee2e2"] * len(row))

    def verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
            "ERROR":   "color: #7c3aed; font-weight: 600",
        }.get(str(val), "color: #64748b")

    df = pd.DataFrame([{
        "Test Case":  r["name"],
        "Expected":   r["expected_verdict"],
        "Actual":     r["actual_verdict"],
        "Exp Unverif":r["expected_violations"],
        "Act Unverif":r["actual_violations"],
        "Result":     "✓" if r["passed"] else "✗",
        "Summary":    r["summary"],
    } for r in rows])

    n_pass    = sum(1 for r in rows if r["passed"])
    n_total   = len(rows)
    acc       = n_pass / n_total * 100 if n_total else 0
    threshold = JUDGE_ACCURACY_THRESHOLD
    cap_status = "PASS ✓" if acc >= threshold else "FAIL ✗"

    _styler = df.style.apply(row_color, axis=1)
    _map = _styler.map
    return (
        _map(verdict_color, subset=["Expected", "Actual"])
        .set_caption(
            f"Judge Calibration — {n_pass}/{n_total} cases fully correct "
            f"({acc:.0f}%) — Threshold {threshold:.0f}% — {cap_status}"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.83rem"), ("padding", "7px 12px"),
                       ("border-bottom", "1px solid #f1f5f9"),
                       ("max-width", "300px"), ("word-wrap", "break-word")]},
        ])
        .hide(axis="index")
    )

display(_style_judge_eval(eval_rows))

## 8. Discussion

The three evaluations form a layered validation hierarchy, each catching failure modes the others cannot.

**Grid search** establishes whether the self-refining loop converges reliably across diverse threat scenarios and model configurations. A ceiling effect — F1=100% across all nine configurations — indicates the corroboration rules fully constrain the output space: the pipeline is rule-limited, not model-limited. When that ceiling is present, the winning configuration is the cheapest one with the lowest average iteration count.

**Ground-truth evaluation** breaks the judge's monopoly on correctness. A lenient judge can emit PASS verdicts on briefs that miss critical correlations. Confidence Accuracy surfaces this — whether the pipeline's relevance ratings are consistent with the evidence weight a human analyst would assign. If Confidence Accuracy falls below threshold, the brief is structurally wrong regardless of the judge's opinion.

**Judge calibration** validates the judge's own reliability before trusting its verdicts in the grid search. If the judge fails calibration — misidentifying rule violations, accepting confidence inflation, or missing cross-source requirements — the F1 scores from the grid search are meaningless. The calibration runs first in any evaluation pipeline for this reason.

**Ceiling interpretation**: When all three evaluations pass with F1=100%, Confidence Accuracy above threshold, and judge accuracy above threshold, the processing stage is operating within its designed constraints. The self-refining loop is working as intended. The appropriate next step is not further tuning of this stage but integration testing across the full intelligence cycle.